In [ ]:
pip install -r requirements.txt

In [3]:
import importlib
import new_config_propmt as config_prompt
importlib.reload(config_prompt)
import evaluate
importlib.reload(evaluate)

<module 'evaluate' from 'c:\\Desktop\\Kusum\\OVGU\\ML_Project_LLM\\fake_reviews_prediction\\evaluate.py'>

In [6]:
import numpy as np
import pandas as pd
from langchain_ollama import OllamaLLM
from langchain_core.prompts import PromptTemplate
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from new_config_propmt import Config
from evaluate import evaluate_model

In [8]:
Config.zero_shot_depth0_width0_ver2

'\n    Role\n    You are an expert at classifying "truthful" and "deceptive" reviews.\n\n    Task\n    Classify the following hotel or product review as either “truthful” or “deceptive.”\n\n    Review\n    "{text}"\n\n    Guidelines:\n    - Deceptive reviews often use generic, exaggerated, or marketing-style language.\n    - Truthful reviews usually contain natural wording and specific details.\n\n    Output Format\n    RESPOND WITH ONE WORD ONLY:\n    truthful or deceptive \n    '

In [ ]:
# Load your CSV
df = pd.read_csv(Config.file_path)
reviews = df["text"].tolist()

In [ ]:
prompting_styles = ['zero_shot_depth0_width0', 'zero_shot_depth0_width1', 'zero_shot_depth1_width0', 'zero_shot_depth2_width1', 'zero_shot_depth3_width1',
                    'one_shot_depth0_width0', 'one_shot_depth0_width1', 'one_shot_depth1_width0', 'one_shot_depth2_width1', 'one_shot_depth3_width1', 
                    'few_shot_depth0_width0', 'few_shot_depth0_width1', 'few_shot_depth1_width0', 'few_shot_depth2_width1', 'few_shot_depth3_width1']

for style in prompting_styles:
    print(f"Evaluating for prompting style: {style}")
    prompt_template = getattr(Config, style)
    
    prompt = PromptTemplate(
        input_variables=["text"],  # the placeholder to be replaced based on prompt template
        template=prompt_template
,    )
    
    # load model and create chain
    llm = OllamaLLM(model=Config.model)
    # Create chain using pipe operator (modern LangChain syntax)
    
    chain = prompt | llm
    
    predictions = []
    for review in reviews:
        response = chain.invoke({"text": review})
        predictions.append(response.strip().lower())

        print(f"Raw output: {response} -> Cleaned: {response.strip().lower()}")
    

    accuracy, f1, conf_matrix = evaluate_model(df, predictions)

    print("=" * 50)
    print(f"{style.upper()} RESULTS")
    print("=" * 50)
    print("Accuracy:", accuracy)
    print("\nF1 Score:", f1)
    print("\nConfusion Matrix:")
    print(conf_matrix)
    print("=" * 50)

In [ ]:
import os
print(os.environ.get("OLLAMA_HOST")) 

os.environ["OLLAMA_HOST"] = "http://localhost:11300"

In [ ]:
# Classify each review
results = []

for r in reviews:
    result = chain.invoke({"text": r})
    raw = result.lower()

    if "deceptive" in raw:
        label = "deceptive"
    elif "truthful" in raw:
        label = "truthful"
    else:
        label = "unknown"  # safety fallback

    print(f"Raw output: {result} -> Cleaned: {label}")
    results.append(label)

In [ ]:
# Classify each review
# results = []
# for r in reviews:
#     result = chain.invoke({"text": r})
#     # Clean the output - extract only "truthful" or "deceptive"
#     label = result.strip().lower()
#     # Extract first word if model adds extra text
#     if " " in label:
#         label = label.split()[0]
#     # Remove any punctuation
#     label = label.strip('.,!?;:')
#     print(f"Raw output: {result} -> Cleaned: {label}")
#     results.append(label)

In [ ]:
accuracy, f1, conf_matrix = evaluate_model(df, results)

print("=" * 50)
print("ZERO-SHOT PROMPTING RESULTS")
print("=" * 50)
print("Accuracy:", accuracy)
print("\nF1 Score:", f1)
print("\nConfusion Matrix:")
print(conf_matrix)
print("=" * 50)

## One-Shot Prompting
Using one example to guide the model's classification

In [ ]:
one_shot_prompt = PromptTemplate(
    input_variables=["text"],
    template = Config.one_shot_prompt_template3
)
# Create one-shot chain
one_shot_chain = one_shot_prompt | llm

In [ ]:
# Classify each review
one_shot_results = []
for r in reviews:
    result = one_shot_chain.invoke({"text": r})
    # Clean the output - extract only "truthful" or "deceptive"
    label = result.strip().lower()
    # Extract first word if model adds extra text
    if " " in label:
        label = label.split()[0]
    # Remove any punctuation
    label = label.strip('.,!?;:')
    print(f"Raw output: {result} -> Cleaned: {label}")
    one_shot_results.append(label)

In [ ]:
accuracy_one_shot, f1_one_shot, conf_matrix_one_shot = evaluate_model(df, one_shot_results)

print("=" * 50)
print("ONE-SHOT PROMPTING RESULTS")
print("=" * 50)
print("Accuracy:", accuracy_one_shot)
print("\nF1 Score:", f1_one_shot)
print("\nConfusion Matrix:")
print(conf_matrix_one_shot)
print("=" * 50)

## Few-Shot Prompting
Using multiple examples to guide the model's classification

In [ ]:
few_shot_prompt = PromptTemplate(
    input_variables=["text"],
    template=Config.few_shot_prompt_template3
)

# Create few-shot chain
few_shot_chain = few_shot_prompt | llm

In [ ]:
# Classify reviews using few-shot prompting
few_shot_results = []
for r in reviews:
    result = few_shot_chain.invoke({"text": r})
    # Clean the output
    label = result.strip().lower()
    if " " in label:
        label = label.split()[0]
    label = label.strip('.,!?;:')
    print(f"Raw output: {result} -> Cleaned: {label}")
    few_shot_results.append(label)

In [ ]:
accuracy_few_shot, f1_few_shot, conf_matrix_few_shot = evaluate_model(df, few_shot_results)

print("=" * 50)
print("FEW-SHOT PROMPTING RESULTS")
print("=" * 50)
print("Accuracy:", accuracy_few_shot)
print("\nF1 Score:", f1_few_shot)
print("\nConfusion Matrix:")
print(conf_matrix_few_shot)
print("=" * 50)

## Compare All Approaches
Compare zero-shot, one-shot, and few-shot results

In [ ]:
# Compare all approaches
comparison_df = pd.DataFrame({
    'Approach': ['Zero-Shot', 'One-Shot', 'Few-Shot'],
    'Accuracy': [accuracy, accuracy_one_shot, accuracy_few_shot],
    'F1 Score': [f1, f1_one_shot, f1_few_shot]
})

print("=" * 60)
print("COMPARISON OF ALL PROMPTING APPROACHES")
print("=" * 60)
print(comparison_df.to_string(index=False))
print("=" * 60)

# Find best approach
best_idx = comparison_df['Accuracy'].idxmax()
best_approach = comparison_df.loc[best_idx, 'Approach']
best_accuracy = comparison_df.loc[best_idx, 'Accuracy']
print(f"\nBest Approach: {best_approach} (Accuracy: {best_accuracy:.4f})")

In [ ]:
import os
print(os.environ.get("OLLAMA_HOST"))